# Omni Claw 二 测试


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

ROOT = Path.cwd()
if ROOT.name == 'test':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print('项目根目录:', ROOT)


项目根目录: d:\code\program\OmniClaw


## 2.1 Provider 调用测试


In [2]:
from backend.app.core.config import get_settings
from backend.app.core.provider import get_provider, ProviderMessage

s = get_settings()
provider = get_provider()

print('provider:', s.provider)
print('model_name:', s.model_name)
print('openai_api_base:', getattr(s, 'openai_api_base', None))
print('openai_api_key set:', bool(getattr(s, 'openai_api_key', None)))
print('provider class:', type(provider).__name__)
assert hasattr(provider, 'generate'), 'provider 缺少 generate 方法'
print('provider 初始化检查通过')


provider: aliyun
model_name: qwen-max
openai_api_base: https://dashscope.aliyuncs.com/compatible-mode/v1
openai_api_key set: True
provider class: OpenAICompatibleProvider
provider 初始化检查通过


如果密钥或网关配置有问题，这个单元会抛错（例如 401）。这是预期行为，可据此排查 `.env` 配置。


In [3]:
try:
    result = provider.generate([
        ProviderMessage(role='user', content='你好，请只回复：ok')
    ])
    print('model output:', result.text)
    assert isinstance(result.text, str) and len(result.text) > 0, '返回内容为空'
    print('真实调用测试通过')
except Exception as e:
    print('真实调用测试失败:', e)
    print('排查建议:')
    print('1) 检查 DEFAULT_PROVIDER / DEFAULT_MODEL')
    print('2) 检查 OPENAI_API_KEY 是否可用')
    print('3) 若使用兼容网关，检查 OPENAI_API_BASE')
    raise


model output: ok
真实调用测试通过


## 2.2 LangGraph 主循环测试


In [4]:
from backend.app.core.agent import AgentState, _route_after_agent, _tools_node

plain_ai = AIMessage(content='普通回复')
state_no_tool: AgentState = {'messages': [HumanMessage(content='你好'), plain_ai]}
route_no_tool = _route_after_agent(state_no_tool)
print('route without tool:', route_no_tool)
assert route_no_tool == 'end'

tool_ai = AIMessage(
    content='我将调用工具',
    tool_calls=[{'id': 'call_1', 'name': 'read_file', 'args': {}}],
)
state_with_tool: AgentState = {'messages': [HumanMessage(content='读取文件'), tool_ai]}
route_with_tool = _route_after_agent(state_with_tool)
print('route with tool:', route_with_tool)
assert route_with_tool == 'tools'

tool_result = _tools_node(state_with_tool)
print('tools node result count:', len(tool_result['messages']))
assert len(tool_result['messages']) == 1
assert isinstance(tool_result['messages'][0], ToolMessage)
assert tool_result['messages'][0].tool_call_id == 'call_1'
assert '暂未注册工具' in tool_result['messages'][0].content

empty_tool_result = _tools_node({'messages': [HumanMessage(content='仅用户消息')]})
assert empty_tool_result == {'messages': []}
print('2.2 LangGraph 主循环测试通过')


route without tool: end
route with tool: tools
tools node result count: 1
2.2 LangGraph 主循环测试通过
